In [1]:
import torch

#device = "cuda:0" if torch.cuda.is_available() else "cpu"
device = "cpu"
device

'cpu'

In [2]:
from imagebind import data
from imagebind.models import imagebind_model
from imagebind.models.imagebind_model import ModalityType

/home/kojo/Code/HGDE/henv/lib/python3.14/site-packages/imagebind/data.py:10: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources
/home/kojo/Code/HGDE/henv/lib/python3.14/site-packages/torch/jit/_script.py:359: FutureWarning: `torch.jit.script_method` is not supported in Python 3.14+ and may break. Please switch to `torch.compile` or `torch.export`.
  warnings.warn(


In [14]:
text_list=["dog.", "car", "bird"]
image_paths=[".assets/dog_image.jpg", ".assets/car_image.jpg", ".assets/bird_image.jpg"]
audio_paths=[".assets/dog_audio.wav", ".assets/car_audio.wav", ".assets/bird_audio.wav"]

In [15]:
model = imagebind_model.imagebind_huge(pretrained=True)
model.eval()
model.to(device)

ImageBindModel(
  (modality_preprocessors): ModuleDict(
    (vision): RGBDTPreprocessor(
      (cls_token): tensor((1, 1, 1280), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Sequential(
          (0): PadIm2Video()
          (1): Conv3d(3, 1280, kernel_size=(2, 14, 14), stride=(2, 14, 14), bias=False)
        )
      )
      (pos_embedding_helper): SpatioTemporalPosEmbeddingHelper(
        (pos_embed): tensor((1, 257, 1280), requires_grad=True)

      )
    )
    (text): TextPreprocessor(
      (pos_embed): tensor((1, 77, 1024), requires_grad=True)
      (mask): tensor((77, 77), requires_grad=False)

      (token_embedding): Embedding(49408, 1024)
    )
    (audio): AudioPreprocessor(
      (cls_token): tensor((1, 1, 768), requires_grad=True)

      (rgbt_stem): PatchEmbedGeneric(
        (proj): Conv2d(1, 768, kernel_size=(16, 16), stride=(10, 10), bias=False)
        (norm_layer): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
      )
 

In [16]:
# Load data
inputs = {
    ModalityType.TEXT: data.load_and_transform_text(text_list, device),
    ModalityType.VISION: data.load_and_transform_vision_data(image_paths, device),
    ModalityType.AUDIO: data.load_and_transform_audio_data(audio_paths, device),
}

inputs

{'text': tensor([[49406,  1929,   269, 49407,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0],
         [49406,  1615, 49407,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
              0,     0,     0,     0,     0,     0, 

In [17]:
import gc

gc.collect()
torch.cuda.empty_cache()

In [18]:
with torch.amp.autocast(device, dtype=torch.float16):
    with torch.no_grad():
        embeddings = model(inputs)

In [19]:
embeddings

{'text': tensor([[ 0.1035, -1.3896,  0.9766,  ...,  1.9639, -4.7109, -1.2051],
         [-0.1189,  1.8018,  0.0282,  ..., -1.7383, -3.2285,  1.5508],
         [ 0.5894, -0.4460,  3.6191,  ...,  0.7100, -1.7930,  1.9365]],
        dtype=torch.float16),
 'vision': tensor([[ 0.0022, -0.0280,  0.0051,  ...,  0.0135,  0.0022,  0.0147],
         [ 0.0260,  0.0055,  0.0398,  ..., -0.0252, -0.0048, -0.0195],
         [ 0.0187,  0.0058,  0.0546,  ...,  0.0043,  0.0097, -0.0093]],
        dtype=torch.float16),
 'audio': tensor([[-0.0287, -0.4927,  1.0059,  ...,  0.3340, -0.1342, -0.6133],
         [ 0.7085,  0.2070, -1.0137,  ...,  0.4949, -0.2336,  0.4712],
         [ 0.2212, -0.2542,  0.3887,  ..., -0.2422,  0.0878,  0.1379]],
        dtype=torch.float16)}

In [20]:
print(
    "Vision x Text: ",
    torch.round(torch.softmax(embeddings[ModalityType.VISION] @ embeddings[ModalityType.TEXT].T, dim=-1), decimals=3),
)
print(
    "Audio x Text: ",
    torch.round(torch.softmax(embeddings[ModalityType.AUDIO] @ embeddings[ModalityType.TEXT].T, dim=-1), decimals=3),
)
print(
    "Vision x Audio: ",
    torch.round(torch.softmax(embeddings[ModalityType.VISION] @ embeddings[ModalityType.AUDIO].T, dim=-1), decimals=3),
)

Vision x Text:  tensor([[0.9980, 0.0020, 0.0000],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.0000, 1.0000]], dtype=torch.float16)
Audio x Text:  tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]], dtype=torch.float16)
Vision x Audio:  tensor([[0.8062, 0.1090, 0.0840],
        [0.1030, 0.7891, 0.1070],
        [0.0020, 0.0020, 0.9961]], dtype=torch.float16)


In [21]:
"""
Vision x Text:  tensor([[0.9980, 0.0020, 0.0000],
        [0.0000, 1.0000, 0.0000],
        [0.0000, 0.0140, 0.9858]], dtype=torch.float16)
Audio x Text:  tensor([[1., 0., 0.],
        [0., 1., 0.],
        [0., 0., 1.]], dtype=torch.float16)
Vision x Audio:  tensor([[0.8062, 0.1090, 0.0840],
        [0.1030, 0.7891, 0.1070],
        [0.0020, 0.0020, 0.9961]], dtype=torch.float16)
"""

'\nVision x Text:  tensor([[0.9980, 0.0020, 0.0000],\n        [0.0000, 1.0000, 0.0000],\n        [0.0000, 0.0140, 0.9858]], dtype=torch.float16)\nAudio x Text:  tensor([[1., 0., 0.],\n        [0., 1., 0.],\n        [0., 0., 1.]], dtype=torch.float16)\nVision x Audio:  tensor([[0.8062, 0.1090, 0.0840],\n        [0.1030, 0.7891, 0.1070],\n        [0.0020, 0.0020, 0.9961]], dtype=torch.float16)\n'